<a href="https://colab.research.google.com/github/khang28082004k-hue/BigData_Lab1/blob/main/A44640_Hadoop_Ecommerce_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Bước 1 – Cài Java và Hadoop
---



In [1]:
!java -version

[0.022s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.022s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [2]:
!wget -q \
  https://dlcdn.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz \
  -O /content/hadoop-3.5.0.tar.gz

!ls -lh /content/hadoop-3.5.0.tar.gz

-rw-r--r-- 1 root root 555M Apr  1  2026 /content/hadoop-3.5.0.tar.gz


In [3]:
!tar -xzf /content/hadoop-3.5.0.tar.gz -C /content/

Bước 2: cấu hình biến môi trường


In [4]:
import os
import subprocess

java_home = subprocess.check_output(
    "dirname $(dirname $(readlink -f $(which java)))",
    shell=True,
    text=True
).strip()

os.environ["JAVA_HOME"] = java_home
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["PATH"] += ":/content/hadoop-3.5.0/bin:/content/hadoop-3.5.0/sbin"

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
HADOOP_HOME = /content/hadoop-3.5.0


In [5]:
import os
hadoop_env = "/content/hadoop-3.5.0/etc/hadoop/hadoop-env.sh"
with open(hadoop_env, "a") as f:
    f.write(f"\nexport JAVA_HOME={os.environ['JAVA_HOME']}\n")
print("Đã cập nhật", hadoop_env)

Đã cập nhật /content/hadoop-3.5.0/etc/hadoop/hadoop-env.sh


In [6]:
!hadoop version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


Phần B - Cấu hình HDFS pseudo-distributed

In [7]:
!mkdir -p /content/hadoop_data/namenode
!mkdir -p /content/hadoop_data/datanode
!ls -ld /content/hadoop_data/*

drwxr-xr-x 2 root root 4096 Oct 14 06:05 /content/hadoop_data/datanode
drwxr-xr-x 2 root root 4096 Oct 14 06:05 /content/hadoop_data/namenode


In [8]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/core-site.xml
<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/core-site.xml


In [9]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml
<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:///content/hadoop_data/namenode</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:///content/hadoop_data/datanode</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml


In [10]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/mapred-site.xml
<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>local</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/mapred-site.xml


In [11]:
!hadoop getconf -confKey fs.defaultFS
!hadoop getconf -confKey dfs.replication
!hadoop getconf -confKey mapreduce.framework.name

ERROR: getconf is not COMMAND nor fully qualified CLASSNAME.
Usage: hadoop [OPTIONS] SUBCOMMAND [SUBCOMMAND OPTIONS]
 or    hadoop [OPTIONS] CLASSNAME [CLASSNAME OPTIONS]
  where CLASSNAME is a user-provided Java class

  OPTIONS is none or any of:

buildpaths                       attempt to add class files from build tree
--config dir                     Hadoop config directory
--debug                          turn on shell script debug mode
--help                           usage information
hostnames list[,of,host,names]   hosts to use in worker mode
hosts filename                   list of hosts to use in worker mode
loglevel level                   set the log4j level for this command
workers                          turn on worker mode

  SUBCOMMAND is one of:


    Admin Commands:

daemonlog     get/set the log level for each daemon

    Client Commands:

archive       create a Hadoop archive
checknative   check native Hadoop and compression libraries availability
classpath     

Phần C - Khởi tạo và khởi động HDFS

In [12]:
!hdfs namenode -format -force

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-14 06:05:44,654 INFO namenode.NameNode: STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = ff307c82eb99/172.28.0.12
STARTUP_MSG:   args = [-format, -force]
STARTUP_MSG:   version = 3.5.0
STARTUP_MSG:   classpath = /content/hadoop-3.5.0/etc/hadoop:/content/hadoop-3.5.0/share/hadoop/common/lib/hadoop-gcp-3.5.0.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/commons-text-1.14.0.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/netty-transport-4.1.130.Final.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/jsr305-3.0.2.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/jakarta.anno

In [13]:
!hdfs --daemon start namenode

In [14]:
!hdfs --daemon start datanode

In [15]:
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2309 NameNode
2373 DataNode
2411 Jps


In [16]:
!hdfs dfsadmin -report

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
report: Call From ff307c82eb99/172.28.0.12 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


In [17]:
!hdfs dfs -ls /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Phần D - Khai thác và sử dụng HDFS

In [18]:
!hdfs dfs -mkdir -p /user/student/input
!hdfs dfs -mkdir -p /user/student/output
!hdfs dfs -ls -R /user/student

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
drwxr-xr-x   

In [19]:
!printf "Hanoi,1200\nDanang,850\nHCM,1700\nHanoi,950\n" > /content/sales.txt
!cat /content/sales.txt

Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [20]:
!hdfs dfs -put /content/sales.txt /user/student/input/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [21]:
!hdfs dfs -ls -h /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items
-rw-r--r--   1 root supergroup         41 2026-10-14 06:06 /user/student/input/sales.txt


In [22]:
!hdfs dfs -cat /user/student/input/sales.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [23]:
!hdfs dfs -head /user/student/input/sales.txt
!hdfs dfs -tail /user/student/input/sales.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [24]:
!hdfs dfs -cp /user/student/input/sales.txt /user/student/input/sales_backup.txt
!hdfs dfs -ls /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 2 items
-rw-r--r--   1 root supergroup         41 2026-10-14 06:06 /user/student/input/sales.txt
-rw-r--r--   1 root supergroup         41 2026-10-14 06:06 /user/student/input/sales_backup.txt


In [25]:
!hdfs dfs -mv /user/student/input/sales_backup.txt /user/student/input/sales_2026.txt
!hdfs dfs -ls /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 2 items
-rw-r--r--   1 root supergroup         41 2026-10-14 06:06 /user/student/input/sales.txt
-rw-r--r--   1 root supergroup         41 2026-10-14 06:06 /user/student/input/sales_2026.txt


In [26]:
!hdfs dfs -du -h /user/student/input
!hdfs dfs -df -h /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
41  41  /user/student/input/sales.txt
41  41  /user/student/input/sales_2026.txt
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Filesystem                Size    Used  Available  Use%
hdfs://localhost:9000  107.7 G  24.1 K     85.4 G    0%


In [27]:
!rm -f /content/sales_downloaded.txt
!hdfs dfs -get /user/student/input/sales.txt /content/sales_downloaded.txt
!cat /content/sales_downloaded.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [28]:
!hdfs dfs -rm /user/student/input/sales_2026.txt
!hdfs dfs -ls /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Deleted /user/student/input/sales_2026.txt
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items
-rw-r--r--   1 root supergroup         41 2026-10-14 06:06 /user/student/input/sales.txt


Phần E - Kiểm tra block và trạng thái HDFS

In [29]:
!hdfs fsck /user/student/input/sales.txt -files -blocks -locations

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&locations=1&path=%2Fuser%2Fstudent%2Finput%2Fsales.txt
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path /user/student/input/sales.txt at Wed Oct 14 06:07:06 UTC 2026

/user/student/input/sales.txt 41 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-710699832-172.28.0.12-1791957946676:blk_1073741825_1001 len=41 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-eab738c1-06af-48e4-9ac5-8943a14875b9,DISK]]


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	41 B
 Total files:	1
 Total

In [30]:
!hdfs fsck /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Connecting to namenode via http://localhost:9870/fsck?ugi=root&path=%2F
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path / at Wed Oct 14 06:07:09 UTC 2026


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			5
 Total symlinks:		0

Replicated Blocks:
 Total size:	41 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 41 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replication:	1.0
 Missing blocks:		0
 Corrupt blocks:		0
 Missing replicas:		0 (0.0 %)
 Blocks queued fo

In [31]:
!hdfs dfsadmin -report

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 91693363252 (85.40 GB)
DFS Remaining: 91693338624 (85.40 GB)
DFS Used: 24628 (24.05 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Badly Distributed Blocks: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Badly Distributed Blocks: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion bl

 Phần F - Chạy MapReduce với dữ liệu nằm trên HDFS

In [32]:
!printf "hadoop big data hadoop mapreduce\nbig data analytics hadoop\n" > /content/words.txt
!hdfs dfs -mkdir -p /user/student/wordcount/input
!hdfs dfs -put -f /content/words.txt /user/student/wordcount/input/
!hdfs dfs -cat /user/student/wordcount/input/words.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
hadoop big da

In [33]:
!hdfs dfs -rm -r -f /user/student/wordcount/output

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [34]:
!hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar wordcount /user/student/wordcount/input /user/student/wordcount/output

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-14 06:07:30,110 INFO input.FileInputFormat: Total input files to process : 1
2026-10-14 06:07:30,154 INFO mapreduce.JobSubmitter: number of splits:1
2026-10-14 06:07:30,509 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1848387112_0001
2026-10-14 06:07:30,511 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-14 06:07:30,763 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-10-14 06:07:30,764 INFO mapreduce.Job: Running job: job_local1848387112_0001
2026-10-14 06:07:30,766 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-10-14 06:07:30,791 INFO output.PathOutputCommitterFactory: No o

In [35]:
!hdfs dfs -cat /user/student/wordcount/output/part-r-00000

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
analytics	1
big	2
data	2
hadoop	3
mapreduce	1


Phần G - Dừng và khởi động lại dịch vụ

In [36]:
!hdfs --daemon stop datanode
!hdfs --daemon stop namenode
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
4415 Jps


In [37]:
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!jps
!hdfs dfs -ls /user/student

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
4459 NameNode
4524 DataNode
4559 Jps
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ls: Call From ff307c82eb99/172.28.0.12 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


13. Bài thực hành tổng hợp đề xuất

In [38]:
import random
random.seed(1)
branches = ["Hanoi","HCM","DaNang","HaiPhong"]
types = ["Deposit","Withdraw","Transfer"]
with open("/content/transactions.txt","w") as f:
    for _ in range(30):
        f.write(f"{random.choice(branches)},{random.choice(types)},{random.randint(100,5000)}\n")
!cat /content/transactions.txt

HCM,Transfer,616
DaNang,Deposit,4158
HaiPhong,Withdraw,3209
HCM,Deposit,4096
Hanoi,Withdraw,3645
Hanoi,Transfer,3748
DaNang,Transfer,1974
Hanoi,Withdraw,350
Hanoi,Deposit,4535
Hanoi,Withdraw,1874
HaiPhong,Transfer,337
HCM,Withdraw,4161
HCM,Withdraw,1991
HCM,Withdraw,2473
Hanoi,Withdraw,4658
Hanoi,Deposit,2528
Hanoi,Transfer,2825
HaiPhong,Transfer,1655
DaNang,Withdraw,4913
HaiPhong,Transfer,3322
Hanoi,Withdraw,2088
HaiPhong,Withdraw,1517
DaNang,Transfer,3169
Hanoi,Withdraw,4265
Hanoi,Deposit,4367
HaiPhong,Withdraw,4111
Hanoi,Withdraw,456
DaNang,Transfer,4959
HaiPhong,Transfer,1495
HCM,Transfer,1959


In [39]:
!hdfs dfs -mkdir -p /user/student/transactions/input
!hdfs dfs -put -f /content/transactions.txt /user/student/transactions/input/
!hdfs dfs -ls /user/student/transactions/input
!hdfs dfs -cat /user/student/transactions/input/transactions.txt

[0.008s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.008s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items

In [40]:
!hdfs fsck /user/student/transactions/input/transactions.txt -files -blocks -locations

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&locations=1&path=%2Fuser%2Fstudent%2Ftransactions%2Finput%2Ftransactions.txt
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path /user/student/transactions/input/transactions.txt at Wed Oct 14 06:10:13 UTC 2026

/user/student/transactions/input/transactions.txt 605 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-710699832-172.28.0.12-1791957946676:blk_1073741829_1005 len=605 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-eab738c1-06af-48e4-9ac5-8943a14875b9,DISK]]


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:

In [41]:
!hdfs dfs -du -h /user/student/transactions/input
!hdfs dfsadmin -report

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
605  605  /user/student/transactions/input/transactions.txt
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 91691868959 (85.39 GB)
DFS Remaining: 91691843584 (85.39 GB)
DFS Used: 25375 (24.78 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing block

In [42]:
open("/content/mapper.py","w").write("""#!/usr/bin/env python3
import sys
for line in sys.stdin:
    p = line.strip().split(',')
    if len(p) == 3:
        print(f"{p[0]}\\t{p[2]}")
""")
open("/content/reducer.py","w").write("""#!/usr/bin/env python3
import sys
cur, total = None, 0
for line in sys.stdin:
    k, v = line.strip().split('\\t')
    if k != cur:
        if cur is not None: print(f"{cur}\\t{total}")
        cur, total = k, 0
    total += int(v)
if cur is not None: print(f"{cur}\\t{total}")
""")
!chmod +x /content/mapper.py /content/reducer.py

!hdfs dfs -rm -r -f /user/student/transactions/output
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files /content/mapper.py,/content/reducer.py \
  -mapper mapper.py -reducer reducer.py \
  -input /user/student/transactions/input \
  -output /user/student/transactions/output
!hdfs dfs -ls /user/student/transactions/output

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
JAR does not exist or is not a normal file: /content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path a

In [43]:
!ls $HADOOP_HOME/share/hadoop/tools/lib/ | grep streaming

hadoop-streaming-3.5.0.jar


In [44]:
!hdfs dfs -rm -r -f /user/student/transactions/output
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar \
  -files /content/mapper.py,/content/reducer.py \
  -mapper mapper.py -reducer reducer.py \
  -input /user/student/transactions/input \
  -output /user/student/transactions/output
!hdfs dfs -ls /user/student/transactions/output

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-14 06:12:30,413 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-10-14 06:12:30,955 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-14 06:12:30,972 INFO mapreduce.JobSubmitter: number of splits:1
2026-10-14 06:12:31,263 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_